In [ ]:
!pip -q install statsmodels scikit-learn joblib

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.statespace.sarimax import SARIMAX
from joblib import dump

warnings.filterwarnings("ignore")
np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)


# LOAD and CLEAN

DATA_PATH = "Final_MasterDataset_upt.csv"
df = pd.read_csv(DATA_PATH)

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", "", regex=False)
        .str.strip()
    )
    rename_map = {
        "PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)": "PM25",
        "Environmental_impact(CO2e/capita)": "CO2e_per_capita",
        "Secondary_SclEnroll_Gross _%": "Secondary_SclEnroll_Gross_Pct",
        "EduExp_GovShare_IMF_%": "EduExp_GovShare_IMF_Pct",
        "Total number of deaths": "Total_Deaths",
        "Asylum Seekers": "Asylum_Seekers",
    }
    return df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

df = clean_columns(df)
df["Year"] = df["Year"].astype(int)
df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

# Drop all NaN columns
all_na_cols = [c for c in df.columns if df[c].isna().all()]
if all_na_cols:
    print("Dropping all-NaN columns:", all_na_cols)
    df = df.drop(columns=all_na_cols)

print("Shape:", df.shape)
print("Countries:", df["Country"].unique())
print("Year range:", df["Year"].min(), "-", df["Year"].max())


# CONFIG (Health only)

TARGET = "Health_Expenditure_GDP"

TRAIN_START, TRAIN_END = 1994, 2016
TEST_START, TEST_END   = 2017, 2022

COUNTRIES = ["USA", "RUS", "CHN", "GBR", "FRA"]

OUTPUT_DIR = Path("arimax_outputs_health_stronger")
OUTPUT_DIR.mkdir(exist_ok=True)


# FEATURE ENGINEERING

# Separate shock dummies (stronger than one combined dummy)
df["D2020"] = (df["Year"] == 2020).astype(int)
df["D2021"] = (df["Year"] == 2021).astype(int)
df["D2022"] = (df["Year"] == 2022).astype(int)

# Base features
BASE_X = [
    "D_Expenditure_GDP",
    "GDP_Growth_Annual",
    "Inflation_Annual",
    "Unemployment_Total_of_TLF",
    "Population",
]

# Health related extras
HEALTH_EXTRAS = [
    "OutOfPocket_Pct_of_CHE",
    "GovHealthExp_Pct_of_CHE",
    "HealthExp_PerCapita_PPP_$",
]

# Keep only existing
BASE_X = [c for c in BASE_X if c in df.columns]
HEALTH_EXTRAS = [c for c in HEALTH_EXTRAS if c in df.columns]

# Lag1 drivers
LAG_BASE = ["D_Expenditure_GDP", "Inflation_Annual", "GDP_Growth_Annual", "Unemployment_Total_of_TLF"]
for col in LAG_BASE:
    if col in df.columns:
        df[f"{col}_lag1"] = df.groupby("Country")[col].shift(1)

LAG_FEATURES = [f"{c}_lag1" for c in LAG_BASE if f"{c}_lag1" in df.columns]

# Final candidate X set
X_COLS_ALL = BASE_X + HEALTH_EXTRAS + LAG_FEATURES + ["D2020", "D2021", "D2022"]
X_COLS_ALL = [c for c in X_COLS_ALL if c in df.columns]

print("Candidate Health X columns:", X_COLS_ALL)


# HELPERS

def split_train_test(country_df: pd.DataFrame):
    train_mask = (country_df["Year"] >= TRAIN_START) & (country_df["Year"] <= TRAIN_END)
    test_mask  = (country_df["Year"] >= TEST_START) & (country_df["Year"] <= TEST_END)
    return country_df.loc[train_mask].copy(), country_df.loc[test_mask].copy()

def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-8, np.nan, y_true)
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100.0

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(y_true) >= 2 else np.nan
    mape = safe_mape(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2}

def mase(y_true, y_pred, y_train):
    """
    Mean Absolute Scaled Error vs naive one-step in training.
    Lower is better. <1 means better than naive.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_train = np.asarray(y_train, dtype=float)

    # naive one-step errors in train
    if len(y_train) < 2:
        return np.nan
    denom = np.mean(np.abs(y_train[1:] - y_train[:-1]))
    if denom < 1e-8:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / denom

def prune_features_by_train(X_train: pd.DataFrame, X_test: pd.DataFrame, max_missing=0.30):
    keep = []
    for c in X_train.columns:
        if X_train[c].isna().mean() > max_missing:
            continue
        if X_train[c].dropna().nunique() <= 1:
            continue
        keep.append(c)
    return X_train[keep].copy(), X_test[keep].copy(), keep

def impute_exog_no_leakage(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """
    No leakage:
      TRAIN: interpolate -> ffill -> fill remaining by TRAIN median
      TEST : ffill using last TRAIN row -> fill remaining by TRAIN median
    """
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()

    X_train_imp = X_train_imp.interpolate(limit_direction="forward").ffill()
    train_medians = X_train_imp.median(numeric_only=True)
    X_train_imp = X_train_imp.fillna(train_medians)

    if len(X_train_imp) > 0 and len(X_test_imp) > 0:
        starter = X_train_imp.iloc[[-1]]
        combined = pd.concat([starter, X_test_imp], axis=0)
        combined = combined.ffill().iloc[1:]
        X_test_imp = combined

    X_test_imp = X_test_imp.fillna(train_medians)
    return X_train_imp, X_test_imp

def clean_target_train_only(y_train: pd.Series):
    """
    Stabilize estimation: train-only interpolation/ffill/bfill.
    Keep as pandas Series (don't convert to numpy).
    """
    y = y_train.astype(float).copy()
    y = y.interpolate(limit_direction="both").ffill().bfill()
    y = y.clip(lower=0)   # <-- keeps Series (instead of np.clip)
    return y

def scale_exog(X_train: pd.DataFrame, X_test: pd.DataFrame):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train.values)
    X_test_s  = scaler.transform(X_test.values)
    return X_train_s, X_test_s, scaler

def select_top_k_features(X_train_imp: pd.DataFrame, y_train_clean: pd.Series, k=5):
    """
    Pick Top-K exog features by absolute correlation with y level and y diff.
    Helps avoid "too many X for too few years".
    """
    if X_train_imp.shape[1] <= k:
        return list(X_train_imp.columns)

    y = y_train_clean.values.astype(float)
    y_diff = np.diff(y) if len(y) >= 3 else None

    scores = {}
    for col in X_train_imp.columns:
        x = X_train_imp[col].values.astype(float)
        # corr with level
        c1 = np.corrcoef(x, y)[0, 1] if np.std(x) > 1e-8 and np.std(y) > 1e-8 else 0.0

        # corr with diff (align lengths)
        if y_diff is not None:
            xd = np.diff(x)
            c2 = np.corrcoef(xd, y_diff)[0, 1] if np.std(xd) > 1e-8 and np.std(y_diff) > 1e-8 else 0.0
        else:
            c2 = 0.0

        score = np.nanmean([abs(c1), abs(c2)])
        if np.isnan(score):
            score = 0.0
        scores[col] = score

    top = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return [c for c, _ in top]

def transform_y(y: np.ndarray, use_log: bool, eps=1e-6):
    y = np.asarray(y, dtype=float)
    if use_log:
        return np.log(y + eps)
    return y

def invert_y(yhat: np.ndarray, use_log: bool, eps=1e-6):
    yhat = np.asarray(yhat, dtype=float)
    if use_log:
        out = np.exp(yhat) - eps
        return np.clip(out, 0, None)
    return yhat

def rolling_one_step_forecast(
    y_train_clean,
    X_train_s: np.ndarray,
    y_test_true: np.ndarray,
    X_test_s: np.ndarray,
    order: tuple,
    trend: str,
    use_log: bool,
    enforce_stationarity=True,
    enforce_invertibility=True,
    eps=1e-6
):
    """
    Rolling 1-step forecast:
      - predict one year
      - append actual y and move forward (no refit)
    """
    # works whether y_train_clean is Series or ndarray
    y0 = transform_y(np.asarray(y_train_clean, dtype=float), use_log, eps=eps)

    model = SARIMAX(
        y0,
        exog=X_train_s,
        order=order,
        trend=trend,
        enforce_stationarity=enforce_stationarity,
        enforce_invertibility=enforce_invertibility
    )
    res = model.fit(disp=False)

    preds = []
    for i in range(len(y_test_true)):
        fc = res.get_forecast(steps=1, exog=X_test_s[i:i+1])

        # predicted_mean may be Series OR ndarray (handle both)
        yhat_t = np.asarray(fc.predicted_mean)[0]

        yhat = invert_y(np.array([yhat_t]), use_log, eps=eps)[0]
        preds.append(float(yhat))

        # update with actual observation
        y_actual = y_test_true[i]
        if np.isnan(y_actual):
            y_append = np.array([np.nan], dtype=float)
        else:
            y_actual = float(max(y_actual, 0.0))
            y_append = transform_y(np.array([y_actual]), use_log, eps=eps)

        res = res.append(endog=y_append, exog=X_test_s[i:i+1], refit=False)

    return np.array(preds, dtype=float), res

def val_rmse_for_spec(
    y_train_clean: pd.Series,
    X_train_s: np.ndarray,
    years_train: np.ndarray,
    order: tuple,
    trend: str,
    use_log: bool,
    val_years=4
):
    """
    Validation = last val_years inside train.
    We do rolling 1-step within train:
      Fit on years < val_start_year, then roll predict each val year with append.
    """
    years_train = np.asarray(years_train)
    uniq = np.sort(np.unique(years_train))
    if len(uniq) <= (val_years + 8):
        return np.inf

    val_start = uniq[-val_years]
    sub_mask = years_train < val_start
    val_mask = years_train >= val_start

    y_sub = y_train_clean.loc[sub_mask]
    y_val = y_train_clean.loc[val_mask].values.astype(float)
    X_sub = X_train_s[sub_mask]
    X_val = X_train_s[val_mask]

    # Need enough points
    if len(y_sub) < 10 or np.sum(~np.isnan(y_val)) < 2:
        return np.inf

    try:
        preds_val, _ = rolling_one_step_forecast(
            y_train_clean=y_sub,
            X_train_s=X_sub,
            y_test_true=y_val,
            X_test_s=X_val,
            order=order,
            trend=trend,
            use_log=use_log,
            enforce_stationarity=True,
            enforce_invertibility=True
        )
        mask = ~np.isnan(y_val)
        rmse = np.sqrt(mean_squared_error(y_val[mask], preds_val[mask]))
        return float(rmse)
    except Exception:
        return np.inf

def select_best_spec(
    y_train_clean: pd.Series,
    X_train_s: np.ndarray,
    years_train: np.ndarray,
    val_years=4
):
    """
    Grid search small specs:
      p,q in 0..2
      d in {0,1}
      trend in {"n","c","ct"}
      use_log in {True, False}
    Choose by minimum validation RMSE (rolling 1-step inside train).
    """
    p_list = [0,1,2]
    q_list = [0,1,2]
    d_list = [0,1]
    trends = ["n", "c", "ct"]
    logs = [True, False]

    best = None
    best_rmse = np.inf

    for use_log in logs:
        for d in d_list:
            for trend in trends:

                for p in p_list:
                    for q in q_list:
                        order = (p,d,q)
                        rmse = val_rmse_for_spec(
                            y_train_clean=y_train_clean,
                            X_train_s=X_train_s,
                            years_train=years_train,
                            order=order,
                            trend=trend,
                            use_log=use_log,
                            val_years=val_years
                        )
                        if rmse < best_rmse:
                            best_rmse = rmse
                            best = (order, trend, use_log)

    if best is None:
        best = ((1,1,1), "c", True)
        best_rmse = np.inf

    return best, best_rmse

def rolling_naive_baseline(y_train_clean: np.ndarray, y_test_true: np.ndarray):
    """
    Rolling naive baseline: yhat_t = last observed value.
    Uses actual y once it becomes available (like a realistic naive forecaster).
    """
    last = float(y_train_clean[-1])
    preds = []
    for i in range(len(y_test_true)):
        preds.append(last)
        if not np.isnan(y_test_true[i]):
            last = float(y_test_true[i])
    return np.array(preds, dtype=float)


# TRAIN + EVAL

def train_evaluate_health_stronger(target: str, X_cols_all: list[str]):
    all_metrics = []
    all_preds = []

    target_dir = OUTPUT_DIR / target
    models_dir = target_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    target_dir.mkdir(parents=True, exist_ok=True)

    for country in COUNTRIES:
        print("\n" + "="*90)
        print(f"Country: {country}")

        cdf = df[df["Country"] == country].copy()
        cdf = cdf[(cdf["Year"] >= TRAIN_START) & (cdf["Year"] <= TEST_END)].sort_values("Year").reset_index(drop=True)

        train_df, test_df = split_train_test(cdf)

        # Target
        y_train_raw = train_df[target]
        y_test = test_df[target].values.astype(float)

        # Train only target cleanup
        y_train_clean = clean_target_train_only(y_train_raw)

        # Exog selection
        use_cols = [c for c in X_cols_all if c in cdf.columns]
        X_train = train_df[use_cols]
        X_test  = test_df[use_cols]

        # Prune too missing / constant
        X_train, X_test, keep_cols = prune_features_by_train(X_train, X_test, max_missing=0.30)
        if len(keep_cols) == 0:
            print("No usable features after pruning. Skipping.")
            continue

        # Impute exog no-leak
        X_train_imp, X_test_imp = impute_exog_no_leakage(X_train, X_test)

        # Top-K feature selection (BIG for performance with short yearly series)
        top_cols = select_top_k_features(X_train_imp, y_train_clean, k=5)
        X_train_imp = X_train_imp[top_cols].copy()
        X_test_imp  = X_test_imp[top_cols].copy()

        # Scale
        X_train_s, X_test_s, scaler = scale_exog(X_train_imp, X_test_imp)

        # Select best spec by validation RMSE (rolling 1-step inside train)
        (best_order, best_trend, best_use_log), best_val_rmse = select_best_spec(
            y_train_clean=y_train_clean,
            X_train_s=X_train_s,
            years_train=train_df["Year"].values,
            val_years=4
        )

        print(f"Chosen spec: order={best_order}, trend='{best_trend}', log={best_use_log}, ValRMSE={best_val_rmse:.4f}")

        # Rolling 1-step forecast on TEST
        try:
            y_pred, res_after_roll = rolling_one_step_forecast(
                y_train_clean=y_train_clean,
                X_train_s=X_train_s,
                y_test_true=y_test,
                X_test_s=X_test_s,
                order=best_order,
                trend=best_trend,
                use_log=best_use_log,
                enforce_stationarity=True,
                enforce_invertibility=True
            )
        except Exception as e:
            print("Rolling forecast failed, falling back to simple model (0,1,0) with drift.")
            best_order, best_trend, best_use_log = (0,1,0), "c", True
            y_pred, res_after_roll = rolling_one_step_forecast(
                y_train_clean=y_train_clean,
                X_train_s=X_train_s,
                y_test_true=y_test,
                X_test_s=X_test_s,
                order=best_order,
                trend=best_trend,
                use_log=best_use_log,
                enforce_stationarity=True,
                enforce_invertibility=True
            )

        # Baseline (rolling naive)
        y_base = rolling_naive_baseline(y_train_clean.values, y_test)

        # Metrics (ignore NaN labels)
        mask = ~np.isnan(y_test)
        metrics = compute_metrics(y_test[mask], y_pred[mask])
        base_metrics = compute_metrics(y_test[mask], y_base[mask])

        improve_rmse = (base_metrics["RMSE"] - metrics["RMSE"]) / max(base_metrics["RMSE"], 1e-8) * 100.0
        mase_val = mase(y_test[mask], y_pred[mask], y_train_clean.values)

        print(
            f"TEST: RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, "
            f"MAPE={metrics['MAPE_%']:.2f}%, R2={metrics['R2']:.4f}, MASE={mase_val:.4f} | "
            f"BaselineRMSE={base_metrics['RMSE']:.4f} | ImproveRMSE={improve_rmse:.2f}%"
        )

        # Save metrics
        all_metrics.append({
            "Target": target,
            "Country": country,
            "Best_Order": str(best_order),
            "Trend": best_trend,
            "Use_Log": bool(best_use_log),
            "Val_RMSE": float(best_val_rmse),
            "Baseline_RMSE": float(base_metrics["RMSE"]),
            "RMSE_Improvement_%": float(improve_rmse),
            "MASE": float(mase_val) if not np.isnan(mase_val) else np.nan,
            **metrics
        })

        # Save predictions
        pred_df = pd.DataFrame({
            "Target": target,
            "Country": country,
            "Year": test_df["Year"].values,
            "y_true": y_test,
            "y_pred": y_pred,
            "y_baseline": y_base
        })
        all_preds.append(pred_df)

        # Save artifacts
        # Save scaler + feature list + chosen spec
        scaler_path = models_dir / f"{country}_scaler.joblib"
        feats_path  = models_dir / f"{country}_features.txt"
        spec_path   = models_dir / f"{country}_spec.txt"

        dump(scaler, scaler_path)
        with open(feats_path, "w") as f:
            f.write("\n".join(top_cols))
        with open(spec_path, "w") as f:
            f.write(f"order={best_order}\ntrend={best_trend}\nuse_log={best_use_log}\nval_rmse={best_val_rmse}\n")


        full_df = cdf[(cdf["Year"] >= TRAIN_START) & (cdf["Year"] <= TEST_END)].copy()
        y_full = full_df[target].astype(float).copy()
        y_full = y_full.interpolate(limit_direction="both").ffill().bfill()
        y_full = np.clip(y_full, 0, None)

        X_full = full_df[top_cols].copy()
        # impute with train medians approach:
        # (use train medians from X_train_imp)
        train_medians = X_train_imp.median(numeric_only=True)
        X_full = X_full.interpolate(limit_direction="forward").ffill().fillna(train_medians)
        X_full_s = scaler.transform(X_full.values)

        eps = 1e-6
        y_full_t = transform_y(y_full.values, best_use_log, eps=eps)

        final_model = SARIMAX(
            y_full_t,
            exog=X_full_s,
            order=best_order,
            trend=best_trend,
            enforce_stationarity=True,
            enforce_invertibility=True
        )
        final_res = final_model.fit(disp=False)
        model_path = models_dir / f"{country}_final_sarimax.pkl"
        final_res.save(model_path)

    metrics_df = pd.DataFrame(all_metrics).sort_values(["Target", "Country"])
    preds_df = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()

    metrics_df.to_csv(target_dir / "metrics_by_country.csv", index=False)
    preds_df.to_csv(target_dir / "predictions_test_2017_2022_rolling.csv", index=False)

    if not metrics_df.empty:
        overall = metrics_df[["MAE","RMSE","MAPE_%","R2","MASE","Baseline_RMSE","RMSE_Improvement_%"]].mean().to_frame().T
        overall.insert(0, "Target", target)
        overall.to_csv(target_dir / "metrics_overall_avg.csv", index=False)

    return metrics_df, preds_df


# RUN

health_metrics, health_preds = train_evaluate_health_stronger(TARGET, X_COLS_ALL)

display(health_metrics)
display(health_preds.head())

print("\nOverall averages (Health):")
summary = health_metrics.groupby("Target")[["MAE","RMSE","MAPE_%","R2","MASE","Baseline_RMSE","RMSE_Improvement_%"]].mean().round(4)
display(summary)


Dropping all-NaN columns: ['Conflict_Intensity']
Shape: (155, 24)
Countries: ['CHN' 'FRA' 'GBR' 'RUS' 'USA']
Year range: 1994 - 2024
Candidate Health X columns: ['D_Expenditure_GDP', 'GDP_Growth_Annual', 'Inflation_Annual', 'Unemployment_Total_of_TLF', 'Population', 'OutOfPocket_Pct_of_CHE', 'GovHealthExp_Pct_of_CHE', 'HealthExp_PerCapita_PPP_$', 'D_Expenditure_GDP_lag1', 'Inflation_Annual_lag1', 'GDP_Growth_Annual_lag1', 'Unemployment_Total_of_TLF_lag1', 'D2020', 'D2021', 'D2022']

Country: USA
Chosen spec: order=(2, 0, 0), trend='ct', log=False, ValRMSE=0.1817
TEST: RMSE=0.5767, MAE=0.3748, MAPE=2.08%, R2=0.5015, MASE=1.7433 | BaselineRMSE=1.1092 | ImproveRMSE=48.01%

Country: RUS
Chosen spec: order=(2, 0, 2), trend='n', log=False, ValRMSE=0.0295
TEST: RMSE=0.8474, MAE=0.7487, MAPE=11.23%, R2=0.2857, MASE=4.6389 | BaselineRMSE=1.0763 | ImproveRMSE=21.27%

Country: CHN
Chosen spec: order=(1, 0, 2), trend='c', log=False, ValRMSE=0.0484
TEST: RMSE=0.1367, MAE=0.0998, MAPE=1.85%, R2=0.34

,Target,Country,Best_Order,Trend,Use_Log,Val_RMSE,Baseline_RMSE,RMSE_Improvement_%,MASE,MAE,RMSE,MAPE_%,R2
2,Health_Expenditure_GDP,CHN,"(1, 0, 2)",c,False,0.048381,0.160530,14.835866,0.812186,0.099842,0.136714,1.846631,0.340078
4,Health_Expenditure_GDP,FRA,"(1, 0, 1)",n,False,0.090120,0.450133,38.260381,2.204186,0.242674,0.277910,2.055923,0.619859
3,Health_Expenditure_GDP,GBR,"(2, 0, 0)",n,False,0.048781,0.920107,7.017332,4.197444,0.601135,0.855540,5.301429,0.286219
1,Health_Expenditure_GDP,RUS,"(2, 0, 2)",n,False,0.029507,1.076333,21.265728,4.638923,0.748719,0.847443,11.225548,0.285710
0,Health_Expenditure_GDP,USA,"(2, 0, 0)",ct,False,0.181714,1.109214,48.008174,1.743300,0.374813,0.576701,2.083035,0.501477


,Target,Country,Year,y_true,y_pred,y_baseline
0,Health_Expenditure_GDP,USA,2017,16.753010,16.732307,16.791143
1,Health_Expenditure_GDP,USA,2018,16.615309,16.648171,16.753010
2,Health_Expenditure_GDP,USA,2019,16.661253,16.647985,16.615309
3,Health_Expenditure_GDP,USA,2020,18.813253,17.593732,16.661253
4,Health_Expenditure_GDP,USA,2021,17.506386,18.134887,18.813253



Overall averages (Health):


,MAE,RMSE,MAPE_%,R2,MASE,Baseline_RMSE,RMSE_Improvement_%
Target,,,,,,,
Health_Expenditure_GDP,0.4134,0.5389,4.5025,0.4067,2.7192,0.7433,25.8775
